# Test the best obtained SimpleHTR model

Inspect a trained `best_model.pth` checkpoint end-to-end, **fully offline**:

1. Export the checkpoint to the ONNX inference artifact + `config.json` (ADR 006).
2. Build a `SimpleHTRModel` from the *local* files (no HF Hub needed).
3. Run recognition on IAM word images and compare PyTorch vs ONNX outputs.
4. Assert numerical equivalence of log-probabilities.

Requires the `training-simple-htr` extra. Run from the repo root with
`uv run jupyter lab`.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import onnxruntime as ort

from xournalpp_htr.inference_models import SimpleHTRModel
from xournalpp_htr.training.simple_htr.dataset import IAM_Words_Dataset
from xournalpp_htr.training.simple_htr.export import export
from xournalpp_htr.training.simple_htr.infer import run_image_through_network

CHECKPOINT = Path("experiments/experiment3/do0.5_augtrue/best_model.pth")
EXPORT_DIR = Path("exports")
assert CHECKPOINT.exists(), f"No checkpoint at {CHECKPOINT.resolve()}"

## 1. Export checkpoint → ONNX + config.json

In [ ]:
paths = export(CHECKPOINT, EXPORT_DIR)
paths

## 2. Build a SimpleHTRModel from the local export

`from_pretrained` would fetch from HF Hub; here we construct the same object
directly from the local ONNX + config so the notebook stays offline.

In [ ]:
with open(paths["config"]) as f:
    config = json.load(f)

model = SimpleHTRModel(
    session=ort.InferenceSession(str(paths["onnx"])),
    config=config,
    revision="local",
)
model

## 3. Load IAM test images

In [ ]:
dataset = IAM_Words_Dataset(
    target_height=config["input_size"]["height"],
    target_width=config["input_size"]["width"],
    augment=False,
)

SAMPLE_INDICES = [0, 100, 500, 1000, 5000]
samples = [(dataset.img_cache[i], dataset.text_cache[i]) for i in SAMPLE_INDICES]
print(f"Loaded {len(samples)} samples: {[s[1] for s in samples]}")

## 4. PyTorch vs ONNX comparison

For each sample, run both inference paths and compare decoded text and raw
log-probabilities.

In [ ]:
session = ort.InferenceSession(str(paths["onnx"]))
input_name = session.get_inputs()[0].name
charset = config["charset"]
norm = config["normalization"]

all_max_diffs = []

for img_raw, gt_text in samples:
    # PyTorch path
    torch_result = run_image_through_network(
        img_raw, model_path=CHECKPOINT, device="cpu"
    )
    torch_text = torch_result["text"]
    torch_log_probs = torch_result["log_probs"]  # (seq_len, num_classes)

    # ONNX path — run session directly to get raw log_probs
    preprocessed = torch_result["model_input_image"]  # reuse same preprocessing
    net_input = preprocessed[None, None, :, :].astype(np.float32)
    onnx_log_probs = session.run(None, {input_name: net_input})[0][:, 0, :]

    # ONNX text via SimpleHTRModel
    onnx_text = model.recognize(img_raw)

    # Numerical comparison
    max_diff = np.max(np.abs(torch_log_probs - onnx_log_probs))
    mean_diff = np.mean(np.abs(torch_log_probs - onnx_log_probs))
    all_max_diffs.append(max_diff)
    text_match = torch_text == onnx_text

    # Visualise
    fig, axes = plt.subplots(1, 3, figsize=(18, 3))
    axes[0].imshow(img_raw, cmap="gray")
    axes[0].set_title(f"Input (GT: '{gt_text}')")
    axes[0].axis("off")
    axes[1].imshow(torch_log_probs.T, aspect="auto", cmap="Blues")
    axes[1].set_title(f"PyTorch: '{torch_text}'")
    axes[1].set_xlabel("Timestep")
    axes[2].imshow(onnx_log_probs.T, aspect="auto", cmap="Blues")
    axes[2].set_title(f"ONNX: '{onnx_text}'")
    axes[2].set_xlabel("Timestep")
    fig.suptitle(
        f"max diff: {max_diff:.2e}, mean diff: {mean_diff:.2e}, "
        f"text match: {text_match}",
        fontsize=11,
    )
    plt.tight_layout()
    plt.show()

    assert text_match, f"Text mismatch: PyTorch='{torch_text}' vs ONNX='{onnx_text}'"

## 5. Summary

In [ ]:
overall_max_diff = max(all_max_diffs)
print(f"Samples tested: {len(samples)}")
print(f"Overall max |PyTorch - ONNX|: {overall_max_diff:.2e}")
print("All texts matched: True")

assert np.allclose(
    torch_log_probs, onnx_log_probs, atol=1e-5
), f"Log-probs differ by more than 1e-5 (max: {overall_max_diff:.2e})"
print("\nONNX export validated successfully.")